# Практична робота №5
## Security monitoring dashboard та підсумковий incident report

У цій фінальній роботі потрібно перетворити результати виявлення підозрілої поведінки з ПР4 на компактний security monitoring dashboard.

Ви не шукаєте аномалії заново. Ваше завдання — узагальнити вже знайдені findings, оцінити ризики, побудувати таблиці для аналізу та сформувати короткий incident report.

### Вхідні дані

- `results/practical_04/detection_findings.csv`
- `results/practical_04/hourly_anomaly_scores.csv`
- `results/practical_04/device_behavior_baseline.csv`
- `results/practical_04/practical_04_summary.json`
- `data/input/device_registry.csv`
- `data/input/metadata.json`

### Результати роботи

Після виконання ноутбука мають бути створені файли:

- `results/practical_05/incident_mart.csv`
- `results/practical_05/device_risk_summary.csv`
- `results/practical_05/scenario_risk_summary.csv`
- `results/practical_05/location_risk_summary.csv`
- `results/practical_05/top_incidents.csv`
- `results/practical_05/practical_05_summary.json`
- `results/practical_05/findings_by_scenario.png`
- `results/practical_05/severity_distribution.png`
- `results/practical_05/top_risky_devices.png`
- `results/practical_05/risk_by_location.png`

> У цій роботі не використовується приватний файл `injected_issues.csv`. Студент працює тільки з результатами власної ПР4.


## Дані студента

Заповніть інформацію про себе.


In [ ]:
STUDENT_NAME = "TODO: Прізвище Ім'я По батькові"
GROUP = "TODO: група"
WORK_DATE = "TODO: дата виконання"


## 1. Підготовка середовища

Цей блок задає шляхи до результатів попередніх практичних і створює каталог для ПР5.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl


In [ ]:
ROOT = Path("/workspace") if Path("/workspace").exists() else Path.cwd().parent

INPUT_DIR = ROOT / "data" / "input"
PRACTICAL_04_RESULTS_DIR = ROOT / "results" / "practical_04"
PRACTICAL_05_RESULTS_DIR = ROOT / "results" / "practical_05"
PRACTICAL_05_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DETECTION_FINDINGS_PATH = PRACTICAL_04_RESULTS_DIR / "detection_findings.csv"
HOURLY_ANOMALY_SCORES_PATH = PRACTICAL_04_RESULTS_DIR / "hourly_anomaly_scores.csv"
DEVICE_BEHAVIOR_BASELINE_PATH = PRACTICAL_04_RESULTS_DIR / "device_behavior_baseline.csv"
PRACTICAL_04_SUMMARY_PATH = PRACTICAL_04_RESULTS_DIR / "practical_04_summary.json"

METADATA_PATH = INPUT_DIR / "metadata.json"
DEVICE_REGISTRY_PATH = INPUT_DIR / "device_registry.csv"

INCIDENT_MART_PATH = PRACTICAL_05_RESULTS_DIR / "incident_mart.csv"
DEVICE_RISK_SUMMARY_PATH = PRACTICAL_05_RESULTS_DIR / "device_risk_summary.csv"
SCENARIO_RISK_SUMMARY_PATH = PRACTICAL_05_RESULTS_DIR / "scenario_risk_summary.csv"
LOCATION_RISK_SUMMARY_PATH = PRACTICAL_05_RESULTS_DIR / "location_risk_summary.csv"
TOP_INCIDENTS_PATH = PRACTICAL_05_RESULTS_DIR / "top_incidents.csv"
SUMMARY_OUTPUT_PATH = PRACTICAL_05_RESULTS_DIR / "practical_05_summary.json"

FINDINGS_BY_SCENARIO_PNG = PRACTICAL_05_RESULTS_DIR / "findings_by_scenario.png"
SEVERITY_DISTRIBUTION_PNG = PRACTICAL_05_RESULTS_DIR / "severity_distribution.png"
TOP_RISKY_DEVICES_PNG = PRACTICAL_05_RESULTS_DIR / "top_risky_devices.png"
RISK_BY_LOCATION_PNG = PRACTICAL_05_RESULTS_DIR / "risk_by_location.png"

ROOT


In [ ]:
required_files = [
    DETECTION_FINDINGS_PATH,
    HOURLY_ANOMALY_SCORES_PATH,
    DEVICE_BEHAVIOR_BASELINE_PATH,
    PRACTICAL_04_SUMMARY_PATH,
    METADATA_PATH,
    DEVICE_REGISTRY_PATH,
]

missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    message = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Практична робота №5 потребує завершеної ПР4. "
        "Не знайдено такі файли:\n" + message
    )

print("Усі потрібні файли знайдено.")


## 2. Завантаження результатів ПР4

У цьому блоці читаються findings, anomaly scores, baseline поведінки пристроїв і metadata варіанта.


In [ ]:
with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

with PRACTICAL_04_SUMMARY_PATH.open("r", encoding="utf-8") as f:
    practical_04_summary = json.load(f)

findings = pl.read_csv(DETECTION_FINDINGS_PATH)
hourly_anomaly_scores = pl.read_csv(HOURLY_ANOMALY_SCORES_PATH)
device_behavior_baseline = pl.read_csv(DEVICE_BEHAVIOR_BASELINE_PATH)
device_registry = pl.read_csv(DEVICE_REGISTRY_PATH)

print(f"Findings з ПР4: {findings.height}")
print(f"Anomaly score rows: {hourly_anomaly_scores.height}")
print(f"Baseline rows: {device_behavior_baseline.height}")
findings.head()


In [ ]:
if findings.is_empty():
    raise AssertionError("detection_findings.csv порожній. Спочатку перевірте ПР4.")

required_finding_columns = [
    "finding_id",
    "scenario_type",
    "device_id",
    "device_type",
    "location",
    "start_ts",
    "end_ts",
    "metric",
    "score",
    "severity",
    "evidence",
]

missing_columns = [column for column in required_finding_columns if column not in findings.columns]
if missing_columns:
    raise AssertionError(f"У detection_findings.csv відсутні колонки: {missing_columns}")

print("Структура detection_findings.csv коректна.")


## 3. Нормалізація severity та risk score

Потрібно перетворити текстову severity на числовий коефіцієнт ризику. Це дасть змогу порівнювати пристрої, сценарії та локації між собою.

Рекомендована шкала:

- `low` → 1
- `medium` → 2
- `high` → 3
- `critical` → 5

Для `risk_score` можна поєднати severity weight і score з ПР4. Якщо `score` відсутній або некоректний, використайте тільки severity weight.


In [ ]:
SEVERITY_WEIGHTS = {
    # TODO: заповніть шкалу severity.
}

def build_incident_mart(findings_df: pl.DataFrame) -> pl.DataFrame:
    """Побудувати фінальну таблицю інцидентів.

    TODO:
    1. Нормалізуйте severity до нижнього регістру.
    2. Додайте `severity_weight`.
    3. Додайте `risk_score`.
    4. Додайте `priority`, наприклад: low / medium / high / urgent.
    5. Поверніть таблицю, відсортовану за спаданням `risk_score`.
    """

    # TODO: реалізуйте incident mart.
    raise NotImplementedError("TODO: implement build_incident_mart")

incident_mart = build_incident_mart(findings)
incident_mart.write_csv(INCIDENT_MART_PATH)
incident_mart.head(10)


Очікувані мінімальні колонки для `incident_mart.csv`. Можна додавати власні додаткові колонки, але ці мають бути присутні.


In [ ]:
INCIDENT_MART_COLUMNS = [
    "finding_id",
    "scenario_type",
    "device_id",
    "device_type",
    "location",
    "start_ts",
    "end_ts",
    "metric",
    "score",
    "severity",
    "severity_weight",
    "risk_score",
    "priority",
    "evidence",
]

missing_columns = [column for column in INCIDENT_MART_COLUMNS if column not in incident_mart.columns]
if missing_columns:
    raise AssertionError(f"incident_mart не містить колонки: {missing_columns}")

if incident_mart.is_empty():
    raise AssertionError("incident_mart порожній.")

print("incident_mart.csv створено.")


## 4. Зведення ризиків по пристроях

Потрібно визначити, які пристрої є найризиковішими. Для цього згрупуйте `incident_mart` за `device_id`, `device_type`, `location` і порахуйте кількість findings, сумарний ризик, максимальну severity та кількість high/critical інцидентів.


In [ ]:
def build_device_risk_summary(incident_mart_df: pl.DataFrame) -> pl.DataFrame:
    """Побудувати зведення ризиків по пристроях.

    TODO:
    1. Згрупуйте інциденти по пристроях.
    2. Порахуйте `findings_count`.
    3. Порахуйте `total_risk_score` і `max_risk_score`.
    4. Порахуйте кількість `high` та `critical` findings.
    5. Відсортуйте пристрої за спаданням ризику.
    """

    # TODO: реалізуйте device risk summary.
    raise NotImplementedError("TODO: implement build_device_risk_summary")

device_risk_summary = build_device_risk_summary(incident_mart)
device_risk_summary.write_csv(DEVICE_RISK_SUMMARY_PATH)
device_risk_summary.head(10)


## 5. Зведення ризиків по сценаріях і локаціях

Тепер потрібно зрозуміти, які типи підозрілої поведінки домінують і в яких локаціях ризик найбільший.


In [ ]:
def build_scenario_risk_summary(incident_mart_df: pl.DataFrame) -> pl.DataFrame:
    """Побудувати зведення по scenario_type."""

    # TODO: згрупуйте по scenario_type і порахуйте findings_count, total_risk_score, avg_risk_score, max_risk_score.
    raise NotImplementedError("TODO: implement build_scenario_risk_summary")


def build_location_risk_summary(incident_mart_df: pl.DataFrame) -> pl.DataFrame:
    """Побудувати зведення по location."""

    # TODO: згрупуйте по location і порахуйте findings_count, affected_devices, total_risk_score, max_risk_score.
    raise NotImplementedError("TODO: implement build_location_risk_summary")

scenario_risk_summary = build_scenario_risk_summary(incident_mart)
location_risk_summary = build_location_risk_summary(incident_mart)

scenario_risk_summary.write_csv(SCENARIO_RISK_SUMMARY_PATH)
location_risk_summary.write_csv(LOCATION_RISK_SUMMARY_PATH)

scenario_risk_summary


In [ ]:
location_risk_summary


## 6. Top incidents для ручної перевірки

Сформуйте короткий список найважливіших findings, які адміністратор мав би перевірити першими.


In [ ]:
TOP_INCIDENTS_LIMIT = 15

def build_top_incidents(incident_mart_df: pl.DataFrame, limit: int = TOP_INCIDENTS_LIMIT) -> pl.DataFrame:
    """Вибрати найважливіші інциденти для ручної перевірки.

    TODO:
    1. Відсортуйте інциденти за `risk_score`.
    2. За однакового ризику пріоритезуйте high/critical severity.
    3. Залиште `limit` рядків.
    """

    # TODO: реалізуйте top incidents.
    raise NotImplementedError("TODO: implement build_top_incidents")

top_incidents = build_top_incidents(incident_mart)
top_incidents.write_csv(TOP_INCIDENTS_PATH)
top_incidents


## 7. Побудова графіків dashboard

Потрібно створити кілька PNG-графіків, які можна вставити у звіт або швидко переглянути як dashboard.

Мінімально потрібно побудувати:

- кількість findings за сценаріями;
- розподіл severity;
- top risky devices;
- risk by location.


In [ ]:
def save_bar_chart(
    df: pl.DataFrame,
    label_column: str,
    value_column: str,
    title: str,
    output_path: Path,
) -> None:
    """Зберегти простий bar chart у PNG.

    TODO:
    1. Перетворіть Polars DataFrame у списки labels і values.
    2. Побудуйте bar chart через matplotlib.
    3. Поверніть підписи осей, title і tight_layout.
    4. Збережіть графік у `output_path`.
    """

    # TODO: реалізуйте допоміжну функцію для графіків.
    raise NotImplementedError("TODO: implement save_bar_chart")


In [ ]:
findings_by_scenario = (
    incident_mart
    .group_by("scenario_type")
    .agg(pl.len().alias("findings_count"))
    .sort("findings_count", descending=True)
)

severity_distribution = (
    incident_mart
    .group_by("severity")
    .agg(pl.len().alias("findings_count"))
    .sort("findings_count", descending=True)
)

top_risky_devices_for_chart = device_risk_summary.head(10)
risk_by_location_for_chart = location_risk_summary.sort("total_risk_score", descending=True)

# TODO: викличте save_bar_chart для кожного з чотирьох графіків.


## 8. Summary JSON

Фінальний JSON потрібен для швидкої перевірки результатів і короткого машинозчитуваного підсумку.


In [ ]:
def first_or_none(df: pl.DataFrame, column: str):
    if df.is_empty() or column not in df.columns:
        return None
    return df.select(column).item(0, 0)

summary = {
    "variant_id": metadata.get("variant_id"),
    "student_id": metadata.get("student_id"),
    "student_name": metadata.get("student_name"),
    "practical_04_findings_total": practical_04_summary.get("findings_total"),
    "incident_mart_rows": int(incident_mart.height),
    "device_risk_rows": int(device_risk_summary.height),
    "scenario_risk_rows": int(scenario_risk_summary.height),
    "location_risk_rows": int(location_risk_summary.height),
    "top_incidents_rows": int(top_incidents.height),
    "highest_risk_device": first_or_none(device_risk_summary, "device_id"),
    "highest_risk_scenario": first_or_none(scenario_risk_summary, "scenario_type"),
    "highest_risk_location": first_or_none(location_risk_summary, "location"),
    "total_risk_score": float(incident_mart.select(pl.sum("risk_score")).item()),
}

with SUMMARY_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary


## 9. Фінальна перевірка результатів

Цей блок перевіряє, що всі очікувані файли створені та мають базову структуру.


In [ ]:
expected_outputs = [
    INCIDENT_MART_PATH,
    DEVICE_RISK_SUMMARY_PATH,
    SCENARIO_RISK_SUMMARY_PATH,
    LOCATION_RISK_SUMMARY_PATH,
    TOP_INCIDENTS_PATH,
    SUMMARY_OUTPUT_PATH,
    FINDINGS_BY_SCENARIO_PNG,
    SEVERITY_DISTRIBUTION_PNG,
    TOP_RISKY_DEVICES_PNG,
    RISK_BY_LOCATION_PNG,
]

for path in expected_outputs:
    if not path.is_file():
        raise AssertionError(f"Не створено файл: {path}")
    if path.stat().st_size == 0:
        raise AssertionError(f"Файл порожній: {path}")

saved_incident_mart = pl.read_csv(INCIDENT_MART_PATH)
missing_columns = [column for column in INCIDENT_MART_COLUMNS if column not in saved_incident_mart.columns]
if missing_columns:
    raise AssertionError(f"incident_mart.csv не містить колонки: {missing_columns}")

print("Фінальна перевірка ПР5 пройдена.")


## 10. Висновок

TODO: напишіть короткий висновок українською мовою.

У висновку потрібно описати:

1. Який пристрій має найвищий ризик і чому.
2. Які scenario types домінують у findings.
3. Які локації виглядають найбільш проблемними.
4. Які top incidents варто перевірити вручну першими.
5. Де можливі false positives або обмеження такого dashboard.
